# Human Labeling Tool for Spot Detection CNN

Interactive widget for labeling image crops for CNN training.

## Workflow
```
1. Generate_Training_Crops.ipynb
   └── Creates: training/training_crops_*/

2. Human_Labeling_Tool.ipynb  ← YOU ARE HERE
   └── Labels crops → training/human_selection_*.npy

3. ML_Tranining_Validation.ipynb
   └── Trains models → spot_detection_cnn.pth
```

## Instructions
1. Set your name in the Configuration cell
2. Run the widget cell
3. Click **Spot Present** or **No Spot** for each crop
4. Run the Save cell when done

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output

## Configuration

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# Your name (for saving annotations)
ANNOTATOR_NAME = 'your_name'  # Change this!

# Folder containing crops to label
CROP_FOLDER = Path('training/training_crops_human_selection')

# Alternative folders:
# CROP_FOLDER = Path('training/training_crops_real_data')

# Output folder for annotations
OUTPUT_FOLDER = Path('training')

# Verify folder exists
if CROP_FOLDER.exists():
    n_crops = len(list(CROP_FOLDER.glob('*.png')))
    print(f"Found {n_crops} crops in {CROP_FOLDER}")
else:
    print(f"Folder not found: {CROP_FOLDER}")

## Load Crops

In [ ]:
# Load all crop images
crop_files = sorted(CROP_FOLDER.glob('*.png'))
list_crops = [np.array(Image.open(f).convert('L')) for f in crop_files]

print(f"Loaded {len(list_crops)} crops")
if list_crops:
    print(f"Crop shape: {list_crops[0].shape}")

## Labeling Widget

In [ ]:
# Storage for labels
global_results = []

def create_labeling_widget(crops):
    """Interactive widget for labeling crops."""
    n_crops = len(crops)
    current_idx = 0
    
    output = widgets.Output()
    progress = widgets.IntProgress(value=0, min=0, max=n_crops, description='Progress:')
    info = widgets.Label(value=f'Crop 1/{n_crops}')
    
    def show_crop(idx):
        crop = crops[idx]
        crop_display = np.array(Image.fromarray(crop).resize((128, 128)))
        crop_display = (crop_display - crop_display.min()) / (crop_display.max() - crop_display.min() + 1e-8)
        
        with output:
            clear_output(wait=True)
            plt.figure(figsize=(4, 4))
            plt.imshow(crop_display, cmap='gray')
            plt.title(f'Crop {idx + 1}')
            plt.axis('off')
            plt.tight_layout()
            plt.show()
        
        info.value = f'Crop {idx + 1}/{n_crops}'
        progress.value = idx
    
    def on_click(is_spot):
        nonlocal current_idx
        if len(global_results) > current_idx:
            global_results[current_idx] = is_spot
        else:
            global_results.append(is_spot)
        
        current_idx += 1
        if current_idx < n_crops:
            show_crop(current_idx)
        else:
            with output:
                clear_output(wait=True)
                print("✅ Labeling complete!")
                print(f"Spots: {sum(global_results)}, No spots: {len(global_results) - sum(global_results)}")
            progress.value = n_crops
    
    def go_back():
        nonlocal current_idx
        if current_idx > 0:
            current_idx -= 1
            show_crop(current_idx)
    
    btn_spot = widgets.Button(description='Spot Present', button_style='success')
    btn_no = widgets.Button(description='No Spot', button_style='danger')
    btn_back = widgets.Button(description='Back', button_style='info')
    
    btn_spot.on_click(lambda _: on_click(True))
    btn_no.on_click(lambda _: on_click(False))
    btn_back.on_click(lambda _: go_back())
    
    display(progress, widgets.HBox([btn_back, btn_spot, btn_no, info]), output)
    show_crop(0)

# Start labeling
if list_crops:
    create_labeling_widget(list_crops)
else:
    print("No crops to label!")

## Save Annotations

In [ ]:
# Save annotations
if global_results:
    labels = np.array(global_results, dtype=bool)
    
    # Pad if incomplete
    if len(labels) < len(list_crops):
        labels = np.pad(labels, (0, len(list_crops) - len(labels)), constant_values=False)
    
    output_file = OUTPUT_FOLDER / f'human_selection_{ANNOTATOR_NAME}.npy'
    np.save(output_file, labels)
    
    print(f"✅ Saved to: {output_file}")
    print(f"Total: {len(labels)} | Spots: {sum(labels)} | No spots: {len(labels) - sum(labels)}")
else:
    print("No annotations to save!")

## Next Steps

After all annotators have labeled:
→ Run `ML_Tranining_Validation.ipynb` to train models